In [1]:
from testsavant.guard import InputGuard, OutputGuard
from testsavant.guard.output_scanners import BanSubstrings
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
long_article = "Once up on a time, in a land far, far away, there lived a young princess named Aurora. She was kind and gentle, with a heart full of love for all living things. One day, while wandering through the forest, she came across a small cottage where"
print(len(long_article))

243


In [3]:
code_example = """
import time, os.path
photofiles = ['img_1074.jpg', 'img_1076.jpg', 'img_1077.jpg']
class BatchRename(Template):
    delimiter = '%'

fmt = input('Enter rename style (%d-date %n-seqnum %f-format):  ')


t = BatchRename(fmt)
date = time.strftime('%d%b%y')
for i, filename in enumerate(photofiles):
    base, ext = os.path.splitext(filename)
    newname = t.substitute(d=date, n=i, f=ext)
    print('{0} --> {1}'.format(filename, newname))
"""

In [4]:
ts_api = InputGuard(
    API_KEY=os.environ.get("TEST_SAVANT_API_KEY"),
    PROJECT_ID=os.environ.get("TEST_SAVANT_PROJECT_ID"),
    remote_addr=os.environ.get("TEST_SAVANT_REMOTE_ADDR")
)

In [5]:
ban_code = BanSubstrings(tag="default", 
                        substrings=["Import time", "import sys"],
                        case_sensitive=True,
                        redact=True,
                        contains_all=False)

In [6]:
ts_api.add_scanner(ban_code)

### short text with code in it

In [8]:
result = ts_api.scan(code_example)
if result.is_valid:
    print("LLM api called")
else:
    print("User request blocked")

LLM api called


In [9]:
result.sanitized_prompt

"\nimport time, os.path\nphotofiles = ['img_1074.jpg', 'img_1076.jpg', 'img_1077.jpg']\nclass BatchRename(Template):\n    delimiter = '%'\n\nfmt = input('Enter rename style (%d-date %n-seqnum %f-format):  ')\n\n\nt = BatchRename(fmt)\ndate = time.strftime('%d%b%y')\nfor i, filename in enumerate(photofiles):\n    base, ext = os.path.splitext(filename)\n    newname = t.substitute(d=date, n=i, f=ext)\n    print('{0} --> {1}'.format(filename, newname))\n"

### a very long article without code

In [10]:
result = ts_api.scan(long_article)
if result.is_valid:
    print("LLM api called")
else:
    print("User request blocked")

LLM api called


### what if we have a very long article with code?

In [11]:
result = ts_api.scan(long_article + code_example)
if result.is_valid:
    print("LLM api called")
else:
    print("User request blocked")

LLM api called
